# Lung & Colon Cancer Histopathology Classification
### Comparative Deep Learning Study: ResNet-50 (CNN) vs. Vision Transformer (ViT-Base) on LC25000 Dataset


## 📌 Project Overview
This project implements and evaluates state-of-the-art Deep Learning architectures for the automated classification of histopathological images from the **LC25000 (Lung and Colon Cancer Histopathological Images)** benchmark dataset across 5 diagnostic categories:
- `colon_aca`: Colon Adenocarcinoma
- `colon_n`: Benign Colon Tissue
- `lung_aca`: Lung Adenocarcinoma
- `lung_n`: Benign Lung Tissue
- `lung_scc`: Lung Squamous Cell Carcinoma


## Lung and Colon Cancer Detection Using Multiple Architectures

### Description

In the second project, use the [LC25000](https://www.kaggle.com/datasets/javaidahmadwani/lc25000) dataset and train it using the following deep learning architectures:
- Vision Transformers (ViTs)
- ResNet

Train each model separately on the dataset, and then compare their performance using metrics such as accuracy, precision, recall, and F1-score to evaluate which model performs best on the task of cancer classification.

In [14]:
# Import Cell

import torch
import torchvision
from pathlib import Path


In [2]:
# Check if CUDA is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if device.type == 'cuda':
    gpu = torch.cuda.get_device_name(0)
    print(f"Using GPU: {gpu}")
    total_memory = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)  # in GB
    print(f"Total GPU Memory: {total_memory:.2f} GB")
    memory_allocated = torch.cuda.memory_allocated(0) / (1024 ** 3)  # in GB
    print(f"Memory Allocated: {memory_allocated:.2f} GB")

Using device: cuda
Using GPU: NVIDIA GeForce MX130
Total GPU Memory: 2.00 GB
Memory Allocated: 0.00 GB


## Dataset Overview

The LC25000 dataset contains histopathological images for lung and colon cancer detection. The dataset is organized into 5 classes:

1. **colon_aca**: Colon adenocarcinoma (malignant)
2. **colon_n**: Normal colon tissue (benign)
3. **lung_aca**: Lung adenocarcinoma (malignant)
4. **lung_n**: Normal lung tissue (benign)
5. **lung_scc**: Lung squamous cell carcinoma (malignant)

The dataset is pre-split into training/validation and test sets. Each image is 768x768 pixels in RGB format. This is a multi-class classification problem where we need to distinguish between different types of cancerous and healthy tissues.

## Required Libraries Installation

We need to install several additional libraries for this project:

- **timm (~50MB)**: PyTorch Image Models library providing pre-trained Vision Transformers and other SOTA models
- **scikit-learn (~30MB)**: For evaluation metrics and data preprocessing utilities
- **matplotlib (~40MB)**: For visualization and plotting training curves
- **seaborn (~5MB)**: Statistical data visualization built on matplotlib
- **Pillow (~10MB)**: Python Imaging Library for image processing
- **tqdm (~1MB)**: Progress bars for training loops

In [15]:
# Install required libraries
!pip install -qq timm scikit-learn matplotlib seaborn Pillow tqdm

In [16]:
# Import all necessary libraries
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchvision
from torchvision import transforms
import timm
from pathlib import Path
import os
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report
from tqdm import tqdm
import time
import random

# Set random seeds for reproducibility
torch.manual_seed(42)
torch.cuda.manual_seed(42)
np.random.seed(42)
random.seed(42)

print("All libraries imported successfully!")

All libraries imported successfully!


## Data Preprocessing and Loading

### Custom Dataset Class
We'll create a custom PyTorch Dataset class to handle the LC25000 dataset. The dataset class will:

1. **Load images**: Read images from the organized folder structure
2. **Apply transformations**: Resize images to 224x224 (standard input size for most pre-trained models)
3. **Normalize**: Apply ImageNet normalization since we're using pre-trained models
4. **Data augmentation**: Apply random transformations to training data to improve generalization

### Data Transformations
- **Training**: Random horizontal flip, random rotation, random resized crop, normalization
- **Validation/Test**: Center crop, resize, normalization (no augmentation for consistent evaluation)

In [5]:
# Custom Dataset Class for LC25000
class LC25000Dataset(Dataset):
    def __init__(self, data_dir, transform=None):
        """
        Args:
            data_dir (string): Directory with all the images organized in class folders
            transform (callable, optional): Optional transform to be applied on a sample
        """
        self.data_dir = Path(data_dir)
        self.transform = transform
        
        # Define class names and create label mapping
        self.classes = ['colon_aca', 'colon_n', 'lung_aca', 'lung_n', 'lung_scc']
        self.class_to_idx = {cls_name: idx for idx, cls_name in enumerate(self.classes)}
        
        # Collect all image paths and labels
        self.images = []
        self.labels = []
        
        for class_name in self.classes:
            class_dir = self.data_dir / class_name
            if class_dir.exists():
                for img_path in class_dir.glob('*.jpeg'):
                    self.images.append(img_path)
                    self.labels.append(self.class_to_idx[class_name])
        
        print(f"Found {len(self.images)} images in {len(self.classes)} classes")
        
        # Print class distribution
        for class_name in self.classes:
            count = self.labels.count(self.class_to_idx[class_name])
            print(f"{class_name}: {count} images")
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()
        
        img_path = self.images[idx]
        image = Image.open(img_path).convert('RGB')
        label = self.labels[idx]
        
        if self.transform:
            image = self.transform(image)
        
        return image, label

# Define data transformations
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("Dataset class and transforms defined successfully!")

Dataset class and transforms defined successfully!


In [6]:
# Create datasets and data loaders
data_root = Path(os.environ.get("LC25000_ROOT", "data/LC25000/lung_colon_image_set"))

# Load training and test datasets
train_dataset = LC25000Dataset(data_root / "Train and Validation Set", transform=train_transform)
test_dataset = LC25000Dataset(data_root / "Test Set", transform=val_test_transform)

# Split training dataset into train and validation (80:20 split)
train_size = int(0.8 * len(train_dataset))
val_size = len(train_dataset) - train_size
train_subset, val_subset = torch.utils.data.random_split(
    train_dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42)
)

# Create validation dataset with appropriate transforms
val_dataset = LC25000Dataset(data_root / "Train and Validation Set", transform=val_test_transform)
val_subset.dataset = val_dataset

print(f"Training samples: {len(train_subset)}")
print(f"Validation samples: {len(val_subset)}")
print(f"Test samples: {len(test_dataset)}")

# Create data loaders
batch_size = 32  # Adjusted for GPU memory constraints
num_workers = 4

train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True)
val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)

print("Data loaders created successfully!")

Found 22501 images in 5 classes
colon_aca: 4500 images
colon_n: 4500 images
lung_aca: 4500 images
lung_n: 4500 images
lung_scc: 4501 images
Found 2499 images in 5 classes
colon_aca: 500 images
colon_n: 500 images
lung_aca: 500 images
lung_n: 500 images
lung_scc: 499 images
Found 22501 images in 5 classes
colon_aca: 4500 images
colon_n: 4500 images
lung_aca: 4500 images
lung_n: 4500 images
lung_scc: 4501 images
Training samples: 18000
Validation samples: 4501
Test samples: 2499
Data loaders created successfully!


## ResNet Architecture

### ResNet Overview
ResNet (Residual Network) is a deep convolutional neural network architecture introduced by He et al. in 2015. It revolutionized deep learning by solving the vanishing gradient problem through **residual connections** (skip connections).

### Key Components:

1. **Residual Blocks**: The core innovation of ResNet
   - Uses skip connections that allow gradients to flow directly through shortcuts
   - Formula: `F(x) = H(x) - x`, where H(x) is the desired mapping and F(x) is the residual
   - Final output: `y = F(x) + x` (element-wise addition)

2. **Architecture Variants**:
   - ResNet-18, ResNet-34: Use basic residual blocks
   - ResNet-50, ResNet-101, ResNet-152: Use bottleneck blocks for efficiency

3. **Bottleneck Block Structure** (ResNet-50):
   - 1×1 conv (reduce dimensions) → 3×3 conv (feature extraction) → 1×1 conv (restore dimensions)
   - Reduces computational complexity while maintaining representational power

### Loss Function: Cross-Entropy Loss
For multi-class classification, we use Cross-Entropy Loss:

**Formula**: `CE = -∑(y_i * log(ŷ_i))`

Where:
- `y_i` is the true label (one-hot encoded)
- `ŷ_i` is the predicted probability for class i

**Advantages**:
- Penalizes wrong predictions more heavily when confidence is high
- Provides stable gradients for backpropagation
- Well-suited for multi-class classification problems

### Why ResNet for Medical Images:
- **Deep feature learning**: Can learn complex hierarchical features in histopathological images
- **Gradient flow**: Skip connections ensure effective training of deep networks
- **Transfer learning**: Pre-trained on ImageNet, can be fine-tuned for medical data

In [7]:
# ResNet Model Implementation
class ResNetModel(nn.Module):
    def __init__(self, num_classes=5, pretrained=True):
        super(ResNetModel, self).__init__()
        
        # Load pre-trained ResNet-50
        self.backbone = torchvision.models.resnet50(pretrained=pretrained)
        
        # Replace the final fully connected layer
        # ResNet-50 has 2048 features in the final layer
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Linear(in_features, num_classes)
        
        # Add dropout for regularization
        self.dropout = nn.Dropout(0.5)
        
    def forward(self, x):
        # Extract features using the backbone (without final FC layer)
        x = self.backbone.conv1(x)
        x = self.backbone.bn1(x)
        x = self.backbone.relu(x)
        x = self.backbone.maxpool(x)
        
        x = self.backbone.layer1(x)
        x = self.backbone.layer2(x)
        x = self.backbone.layer3(x)
        x = self.backbone.layer4(x)
        
        # Global average pooling
        x = self.backbone.avgpool(x)
        x = torch.flatten(x, 1)
        
        # Apply dropout and final classification layer
        x = self.dropout(x)
        x = self.backbone.fc(x)
        
        return x

# Create ResNet model
resnet_model = ResNetModel(num_classes=5, pretrained=True).to(device)

# Print model summary
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"ResNet Model:")
print(f"Total trainable parameters: {count_parameters(resnet_model):,}")
print("Model created successfully!")

/home/msd/anaconda3/envs/py12/lib/python3.13/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/msd/anaconda3/envs/py12/lib/python3.13/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /home/msd/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100%|██████████████████████████████████████████████████████████████████████████████| 97.8M/97.8M [01:20<00:00, 1.28MB/s]


ResNet Model:
Total trainable parameters: 23,518,277
Model created successfully!


## Vision Transformer (ViT) Architecture

### Vision Transformer Overview
Vision Transformer (ViT) was introduced by Dosovitskiy et al. in 2020, adapting the Transformer architecture from NLP to computer vision. It treats images as sequences of patches, similar to how text is treated as sequences of tokens.

### Key Components:

1. **Image Patch Embedding**:
   - Input image (224×224) is divided into fixed-size patches (typically 16×16)
   - Each patch is flattened and linearly projected to embedding dimension
   - Positional embeddings are added to retain spatial information

2. **Transformer Encoder**:
   - **Multi-Head Self-Attention (MSA)**:
     - Formula: `Attention(Q,K,V) = softmax(QK^T/√d_k)V`
     - Allows patches to attend to all other patches in the image
     - Captures long-range dependencies effectively
   
   - **Multi-Layer Perceptron (MLP)**:
     - Two linear layers with GELU activation
     - Provides non-linear transformations

3. **Architecture Details**:
   - Layer Normalization (LN) applied before each sub-layer (Pre-LN)
   - Residual connections around each sub-layer
   - Classification token [CLS] prepended to patch sequence

### Mathematical Formulation:

**Patch Embedding**: `z_0 = [x_{class}; x_p^1E; x_p^2E; ...; x_p^NE] + E_{pos}`

**Transformer Layer**: 
```
z'_l = MSA(LN(z_{l-1})) + z_{l-1}
z_l = MLP(LN(z'_l)) + z'_l
```

**Classification**: `y = LN(z_L^0)`

### Loss Function: Cross-Entropy Loss (Same as ResNet)
The same cross-entropy loss is used, but ViT's attention mechanism provides different inductive biases compared to CNNs.

### Advantages for Medical Images:
- **Global context**: Self-attention captures long-range spatial relationships
- **No locality bias**: Unlike CNNs, ViT doesn't assume local connectivity
- **Interpretability**: Attention maps can visualize which regions the model focuses on
- **Scalability**: Performance improves with larger datasets and model sizes

In [8]:
# Vision Transformer Model Implementation
class ViTModel(nn.Module):
    def __init__(self, num_classes=5, pretrained=True):
        super(ViTModel, self).__init__()
        
        # Load pre-trained Vision Transformer from timm library
        # Using ViT-Base with 16x16 patches trained on ImageNet
        self.backbone = timm.create_model('vit_base_patch16_224', pretrained=pretrained)
        
        # Replace the head (classification layer)
        # ViT-Base has 768 features in the final layer
        in_features = self.backbone.head.in_features
        self.backbone.head = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(in_features, num_classes)
        )
        
    def forward(self, x):
        return self.backbone(x)

# Create Vision Transformer model
vit_model = ViTModel(num_classes=5, pretrained=True).to(device)

print(f"Vision Transformer Model:")
print(f"Total trainable parameters: {count_parameters(vit_model):,}")
print("Model created successfully!")

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Vision Transformer Model:
Total trainable parameters: 85,802,501
Model created successfully!


## Training and Evaluation Functions

### Training Strategy
We'll implement a comprehensive training pipeline with the following components:

1. **Loss Function**: CrossEntropyLoss for multi-class classification
2. **Optimizer**: Adam with learning rate scheduling
3. **Learning Rate Schedule**: ReduceLROnPlateau to adapt learning rate based on validation performance
4. **Early Stopping**: Prevent overfitting by stopping when validation performance plateaus
5. **Model Checkpointing**: Save best model based on validation accuracy

### Evaluation Metrics
We'll compute comprehensive metrics for model comparison:

- **Accuracy**: Overall correct predictions / total predictions
- **Precision**: True Positives / (True Positives + False Positives) - per class and macro-averaged
- **Recall**: True Positives / (True Positives + False Negatives) - per class and macro-averaged  
- **F1-Score**: Harmonic mean of precision and recall - per class and macro-averaged
- **Confusion Matrix**: Detailed breakdown of predictions vs actual classes

In [1]:
# Training and Evaluation Functions

def train_model(model, train_loader, val_loader, num_epochs=25, learning_rate=1e-4, patience=7):
    """
    Train a model with early stopping and learning rate scheduling
    """
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3, verbose=True)
    
    # Training history
    train_losses = []
    train_accuracies = []
    val_losses = []
    val_accuracies = []
    
    best_val_acc = 0.0
    patience_counter = 0
    best_model_state = None
    
    print(f"Starting training for {num_epochs} epochs...")
    
    for epoch in range(num_epochs):
        start_time = time.time()
        
        # Training phase
        model.train()
        running_loss = 0.0
        correct_predictions = 0
        total_samples = 0
        
        train_pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs} [Train]')
        for batch_idx, (images, labels) in enumerate(train_pbar):
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total_samples += labels.size(0)
            correct_predictions += (predicted == labels).sum().item()
            
            # Update progress bar
            train_pbar.set_postfix({
                'Loss': f'{running_loss/(batch_idx+1):.4f}',
                'Acc': f'{100.*correct_predictions/total_samples:.2f}%'
            })
        
        train_loss = running_loss / len(train_loader)
        train_acc = correct_predictions / total_samples
        
        # Validation phase
        model.eval()
        val_loss = 0.0
        correct = 0
        total = 0
        
        with torch.no_grad():
            val_pbar = tqdm(val_loader, desc=f'Epoch {epoch+1}/{num_epochs} [Val]')
            for images, labels in val_pbar:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                
                val_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
                
                val_pbar.set_postfix({
                    'Loss': f'{val_loss/(len(val_loader)):.4f}',
                    'Acc': f'{100.*correct/total:.2f}%'
                })
        
        val_loss /= len(val_loader)
        val_acc = correct / total
        
        # Update learning rate scheduler
        scheduler.step(val_acc)
        
        # Save training history
        train_losses.append(train_loss)
        train_accuracies.append(train_acc)
        val_losses.append(val_loss)
        val_accuracies.append(val_acc)
        
        # Print epoch results
        epoch_time = time.time() - start_time
        print(f'Epoch {epoch+1}/{num_epochs}:')
        print(f'  Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}')
        print(f'  Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}')
        print(f'  Time: {epoch_time:.2f}s')
        print()
        
        # Early stopping and best model saving
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_state = model.state_dict().copy()
            patience_counter = 0
            print(f'  New best validation accuracy: {best_val_acc:.4f}')
        else:
            patience_counter += 1
            
        if patience_counter >= patience:
            print(f'Early stopping triggered after {epoch+1} epochs')
            break
    
    # Load best model
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        print(f'Loaded best model with validation accuracy: {best_val_acc:.4f}')
    
    return {
        'train_losses': train_losses,
        'train_accuracies': train_accuracies,
        'val_losses': val_losses,
        'val_accuracies': val_accuracies,
        'best_val_acc': best_val_acc
    }

def evaluate_model(model, test_loader, class_names):
    """
    Evaluate model on test set and return comprehensive metrics
    """
    model.eval()
    all_predictions = []
    all_labels = []
    
    with torch.no_grad():
        test_pbar = tqdm(test_loader, desc='Testing')
        for images, labels in test_pbar:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            
            all_predictions.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    # Calculate metrics
    accuracy = accuracy_score(all_labels, all_predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_predictions, average='macro')
    
    # Per-class metrics
    precision_per_class, recall_per_class, f1_per_class, _ = precision_recall_fscore_support(
        all_labels, all_predictions, average=None, labels=range(len(class_names))
    )
    
    # Confusion matrix
    cm = confusion_matrix(all_labels, all_predictions)
    
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'precision_per_class': precision_per_class,
        'recall_per_class': recall_per_class,
        'f1_per_class': f1_per_class,
        'confusion_matrix': cm,
        'predictions': all_predictions,
        'labels': all_labels
    }

print("Training and evaluation functions defined successfully!")

Training and evaluation functions defined successfully!


In [2]:
# Visualization Functions

def plot_training_history(history, model_name):
    """
    Plot training and validation loss and accuracy curves
    """
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    
    # Plot losses
    ax1.plot(history['train_losses'], label='Training Loss', color='blue', linewidth=2)
    ax1.plot(history['val_losses'], label='Validation Loss', color='red', linewidth=2)
    ax1.set_title(f'{model_name} - Training and Validation Loss')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Plot accuracies
    ax2.plot(history['train_accuracies'], label='Training Accuracy', color='blue', linewidth=2)
    ax2.plot(history['val_accuracies'], label='Validation Accuracy', color='red', linewidth=2)
    ax2.set_title(f'{model_name} - Training and Validation Accuracy')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

def plot_confusion_matrix(cm, class_names, model_name):
    """
    Plot confusion matrix with proper formatting
    """
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=class_names, yticklabels=class_names,
                cbar_kws={'label': 'Count'})
    plt.title(f'{model_name} - Confusion Matrix')
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.xticks(rotation=45)
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

def plot_metrics_comparison(resnet_metrics, vit_metrics, class_names):
    """
    Compare metrics between ResNet and ViT models
    """
    metrics = ['precision_per_class', 'recall_per_class', 'f1_per_class']
    metric_names = ['Precision', 'Recall', 'F1-Score']
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    for i, (metric, name) in enumerate(zip(metrics, metric_names)):
        x = np.arange(len(class_names))
        width = 0.35
        
        axes[i].bar(x - width/2, resnet_metrics[metric], width, label='ResNet', alpha=0.8, color='skyblue')
        axes[i].bar(x + width/2, vit_metrics[metric], width, label='ViT', alpha=0.8, color='lightcoral')
        
        axes[i].set_xlabel('Classes')
        axes[i].set_ylabel(name)
        axes[i].set_title(f'Per-Class {name} Comparison')
        axes[i].set_xticks(x)
        axes[i].set_xticklabels(class_names, rotation=45)
        axes[i].legend()
        axes[i].grid(True, alpha=0.3)
        axes[i].set_ylim(0, 1)
    
    plt.tight_layout()
    plt.show()

def print_detailed_results(metrics, model_name, class_names):
    """
    Print detailed evaluation results
    """
    print(f"\n{model_name} - Detailed Results:")
    print("=" * 50)
    print(f"Overall Accuracy: {metrics['accuracy']:.4f}")
    print(f"Macro-averaged Precision: {metrics['precision']:.4f}")
    print(f"Macro-averaged Recall: {metrics['recall']:.4f}")
    print(f"Macro-averaged F1-Score: {metrics['f1']:.4f}")
    
    print("\nPer-Class Results:")
    print("-" * 30)
    for i, class_name in enumerate(class_names):
        print(f"{class_name}:")
        print(f"  Precision: {metrics['precision_per_class'][i]:.4f}")
        print(f"  Recall: {metrics['recall_per_class'][i]:.4f}")
        print(f"  F1-Score: {metrics['f1_per_class'][i]:.4f}")
        print()

print("Visualization functions defined successfully!")

Visualization functions defined successfully!


## Model Saving and Loading for Colab-to-Local Workflow

### For Colab Training:
Since you'll be training on Colab with T4 GPU, we'll save the trained models to a `models` directory that you can download and use locally.

### For Local Inference:
We'll implement memory-efficient loading and inference functions that work well with various hardware configurations including limited GPU memory.

## Model Saving and Loading

### Model Persistence
After training, we'll save the trained models for later use:

- **Model weights**: State dictionaries containing trained parameters
- **Training history**: Loss and accuracy curves for analysis
- **Metadata**: Model information including architecture and performance

### Usage Notes:
- Models are saved in the `models/` directory
- Compatible with different hardware setups (CPU/GPU)
- Can be loaded for inference or further training

In [3]:
# Model Saving and Loading Functions
import os
import json

def save_model_and_history(model, history, model_name, save_dir="models"):
    """
    Save trained model, history, and metadata
    """
    os.makedirs(save_dir, exist_ok=True)
    
    # Save model state dict
    model_path = os.path.join(save_dir, f"{model_name}_model.pth")
    torch.save(model.state_dict(), model_path)
    print(f"Model weights saved to: {model_path}")
    
    # Save training history
    history_path = os.path.join(save_dir, f"{model_name}_history.json")
    with open(history_path, 'w') as f:
        json.dump(history, f, indent=2)
    print(f"Training history saved to: {history_path}")
    
    # Save model metadata
    metadata = {
        'model_name': model_name,
        'num_classes': 5,
        'class_names': ['colon_aca', 'colon_n', 'lung_aca', 'lung_n', 'lung_scc'],
        'input_size': 224,
        'best_val_acc': history.get('best_val_acc', 0.0),
        'total_parameters': count_parameters(model)
    }
    
    metadata_path = os.path.join(save_dir, f"{model_name}_metadata.json")
    with open(metadata_path, 'w') as f:
        json.dump(metadata, f, indent=2)
    print(f"Model metadata saved to: {metadata_path}")
    
    return model_path, history_path, metadata_path

def load_model_for_inference(model_path, metadata_path, model_type="resnet", device=None):
    """
    Load trained model for inference
    """
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # Load metadata
    with open(metadata_path, 'r') as f:
        metadata = json.load(f)
    
    print(f"Loading {metadata['model_name']} model...")
    print(f"Parameters: {metadata['total_parameters']:,}")
    print(f"Best validation accuracy: {metadata['best_val_acc']:.4f}")
    
    # Create model based on type
    if model_type.lower() == "resnet":
        model = ResNetModel(num_classes=metadata['num_classes'], pretrained=False)
    elif model_type.lower() == "vit":
        model = ViTModel(num_classes=metadata['num_classes'], pretrained=False)
    else:
        raise ValueError("model_type must be 'resnet' or 'vit'")
    
    # Load state dict
    try:
        state_dict = torch.load(model_path, map_location=device)
        model.load_state_dict(state_dict)
        model.to(device)
        model.eval()
        print(f"Model loaded successfully on {device}")
        
        return model, metadata
    except Exception as e:
        print(f"Error loading model: {e}")
        return None, None

print("Model saving and loading functions defined successfully!")

Model saving and loading functions defined successfully!


## Training ResNet Model

Now we'll train the ResNet-50 model on our lung and colon cancer dataset. The training process includes:

- **Learning Rate**: 1e-4 (suitable for fine-tuning pre-trained models)
- **Batch Size**: 32 (adjusted for GPU memory constraints)  
- **Epochs**: 25 with early stopping (patience=7)
- **Optimization**: Adam optimizer with weight decay for regularization
- **Scheduler**: ReduceLROnPlateau to adaptively reduce learning rate

The model will be saved automatically when validation accuracy improves.

In [ ]:
# Train ResNet Model
print("Starting ResNet Training...")
print("=" * 60)

# Define class names for reference
class_names = ['colon_aca', 'colon_n', 'lung_aca', 'lung_n', 'lung_scc']

# Train ResNet model
resnet_history = train_model(
    model=resnet_model,
    train_loader=train_loader,
    val_loader=val_loader,
    num_epochs=25,
    learning_rate=1e-4,
    patience=7
)

print("ResNet training completed!")
print(f"Best validation accuracy: {resnet_history['best_val_acc']:.4f}")

# Save ResNet model and history
print("\nSaving ResNet model...")
save_model_and_history(resnet_model, resnet_history, "ResNet50")

# Plot training history
plot_training_history(resnet_history, "ResNet-50")

Starting ResNet Training...


/home/msd/anaconda3/envs/py12/lib/python3.13/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Starting training for 25 epochs...


Epoch 1/25 [Train]:   0%|                                                                       | 0/563 [00:00<?, ?it/s]

## Training Vision Transformer Model

Now we'll train the Vision Transformer model with the same hyperparameters for fair comparison:

- **Learning Rate**: 1e-4 (suitable for fine-tuning pre-trained models)
- **Batch Size**: 32 (adjusted for GPU memory constraints)
- **Epochs**: 25 with early stopping (patience=7)
- **Optimization**: Adam optimizer with weight decay for regularization
- **Scheduler**: ReduceLROnPlateau to adaptively reduce learning rate

Note: ViT typically requires more computational resources and may take longer to train than ResNet.

In [10]:
import torch
print(torch.__version__)

2.6.0+cu126


In [ ]:
# Train Vision Transformer Model
print("Starting Vision Transformer Training...")
print("=" * 60)

# Train ViT model
vit_history = train_model(
    model=vit_model,
    train_loader=train_loader,
    val_loader=val_loader,
    num_epochs=25,
    learning_rate=1e-4,
    patience=7
)

print("Vision Transformer training completed!")
print(f"Best validation accuracy: {vit_history['best_val_acc']:.4f}")

# Save ViT model and history
print("\nSaving Vision Transformer model...")
save_model_and_history(vit_model, vit_history, "ViT_Base")

# Plot training history
plot_training_history(vit_history, "Vision Transformer")

In [4]:
# Training Functions
def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, 
                num_epochs=50, device=None, save_dir="models", model_name="model"):
    """
    Train a model with validation and automatic saving
    """
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    model.to(device)
    print(f"Training on device: {device}")
    
    # Training history
    history = {
        'train_loss': [],
        'train_acc': [],
        'val_loss': [],
        'val_acc': [],
        'best_val_acc': 0.0,
        'best_epoch': 0
    }
    
    best_val_acc = 0.0
    patience = 10
    patience_counter = 0
    
    print(f"Starting training for {num_epochs} epochs...")
    print(f"Early stopping patience: {patience} epochs")
    print("-" * 60)
    
    for epoch in range(num_epochs):
        start_time = time.time()
        
        # Training phase
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0
        
        train_progress = tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs} [Train]')
        
        for batch_idx, (data, targets) in enumerate(train_progress):
            data, targets = data.to(device), targets.to(device)
            
            optimizer.zero_grad()
            outputs = model(data)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            train_total += targets.size(0)
            train_correct += (predicted == targets).sum().item()
            
            # Update progress bar
            train_acc = 100. * train_correct / train_total
            train_progress.set_postfix({
                'Loss': f'{loss.item():.4f}',
                'Acc': f'{train_acc:.2f}%'
            })
        
        # Validation phase
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():
            val_progress = tqdm(val_loader, desc=f'Epoch {epoch+1}/{num_epochs} [Val]  ')
            
            for data, targets in val_progress:
                data, targets = data.to(device), targets.to(device)
                outputs = model(data)
                loss = criterion(outputs, targets)
                
                val_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                val_total += targets.size(0)
                val_correct += (predicted == targets).sum().item()
                
                val_acc = 100. * val_correct / val_total
                val_progress.set_postfix({
                    'Loss': f'{loss.item():.4f}',
                    'Acc': f'{val_acc:.2f}%'
                })
        
        # Calculate epoch metrics
        train_loss = train_loss / len(train_loader)
        train_acc = 100. * train_correct / train_total
        val_loss = val_loss / len(val_loader)
        val_acc = 100. * val_correct / val_total
        
        # Update history
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        
        # Learning rate scheduling
        if scheduler:
            scheduler.step(val_loss)
        
        # Check for best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            history['best_val_acc'] = best_val_acc
            history['best_epoch'] = epoch + 1
            patience_counter = 0
            
            # Save best model
            save_model_and_history(model, history, f"{model_name}_best", save_dir)
        else:
            patience_counter += 1
        
        # Print epoch summary
        epoch_time = time.time() - start_time
        print(f"\nEpoch {epoch+1}/{num_epochs} Summary:")
        print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
        print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")
        print(f"Best Val Acc: {best_val_acc:.2f}% (Epoch {history['best_epoch']})")
        print(f"Time: {epoch_time:.2f}s, Patience: {patience_counter}/{patience}")
        print("-" * 60)
        
        # Early stopping
        if patience_counter >= patience:
            print(f"Early stopping triggered after {epoch+1} epochs")
            print(f"Best validation accuracy: {best_val_acc:.2f}% at epoch {history['best_epoch']}")
            break
    
    # Save final model
    save_model_and_history(model, history, f"{model_name}_final", save_dir)
    
    return model, history

print("Training function defined successfully!")

Training function defined successfully!


In [ ]:
# Local Loading Functions for Any GPU/CPU Setup

def load_model_for_inference(model_path, metadata_path, model_type="resnet", device="cpu"):
    """
    Load trained model for inference on weak GPU/CPU
    Optimized for various hardware constraints
    """
    # Load metadata
    with open(metadata_path, 'r') as f:
        metadata = json.load(f)
    
    print(f"Loading {metadata['model_name']} model...")
    print(f"Parameters: {metadata['total_parameters']:,}")
    print(f"Best validation accuracy: {metadata['best_val_acc']:.4f}")
    
    # Create model based on type
    if model_type.lower() == "resnet":
        model = ResNetModel(num_classes=metadata['num_classes'], pretrained=False)
    elif model_type.lower() == "vit":
        model = ViTModel(num_classes=metadata['num_classes'], pretrained=False)
    else:
        raise ValueError("model_type must be 'resnet' or 'vit'")
    
    # Load state dict
    try:
        state_dict = torch.load(model_path, map_location=device)
        model.load_state_dict(state_dict)
        model.to(device)
        model.eval()
        print(f"Model loaded successfully on {device}")
        
        # Clear cache if using GPU
        if device != "cpu" and torch.cuda.is_available():
            torch.cuda.empty_cache()
            
        return model, metadata
    except Exception as e:
        print(f"Error loading model: {e}")
        return None, None

def memory_efficient_inference(model, data_loader, device="cpu", batch_size_override=None):
    """
    Memory-efficient inference for weak GPU
    """
    if batch_size_override:
        # Create new dataloader with smaller batch size
        dataset = data_loader.dataset
        data_loader = DataLoader(dataset, batch_size=batch_size_override, shuffle=False, num_workers=2)
    
    model.eval()
    all_predictions = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels in tqdm(data_loader, desc='Inference'):
            # Move to device in smaller chunks if needed
            images = images.to(device)
            labels = labels.to(device)
            
            # Forward pass
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            
            # Move back to CPU immediately to save GPU memory
            all_predictions.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
            # Clear GPU cache after each batch
            if device != "cpu" and torch.cuda.is_available():
                torch.cuda.empty_cache()
    
    return all_predictions, all_labels

def quick_performance_check(model, test_loader, metadata, device="cpu"):
    """
    Quick performance check for loaded model
    """
    print(f"\nPerforming quick accuracy check on {device}...")
    
    # Use smaller batch size for weak GPU
    batch_size = 16 if device == "cpu" else 8
    predictions, labels = memory_efficient_inference(model, test_loader, device, batch_size)
    
    accuracy = accuracy_score(labels, predictions)
    print(f"Test Accuracy: {accuracy:.4f}")
    print(f"Expected Accuracy: {metadata['best_val_acc']:.4f}")
    
    return accuracy

print("Local loading functions defined successfully!")

Local loading functions defined successfully!


In [7]:
# Generalized Model Management
import os
import shutil

def create_model_summary():
    """
    Create summary of trained models and results
    """
    print("Creating model summary...")
    
    # Show available models
    models_dir = "models"
    if os.path.exists(models_dir):
        print(f"\nSaved files in {models_dir} directory:")
        for file in os.listdir(models_dir):
            file_path = os.path.join(models_dir, file)
            file_size = os.path.getsize(file_path) / (1024 * 1024)  # MB
            print(f"  {file} ({file_size:.1f} MB)")
    else:
        print("No models directory found. Train models first.")

def package_trained_models():
    """
    Package all trained models and results into a zip file
    """
    import zipfile
    
    models_dir = "models"
    if not os.path.exists(models_dir):
        print("No models directory found. Train models first.")
        return
    
    # Create zip file with all model files
    zip_filename = "trained_models.zip"
    with zipfile.ZipFile(zip_filename, 'w') as zipf:
        for root, dirs, files in os.walk(models_dir):
            for file in files:
                file_path = os.path.join(root, file)
                arcname = os.path.relpath(file_path, ".")
                zipf.write(file_path, arcname)
    
    print(f"All trained models packaged into {zip_filename}")
    print(f"Package size: {os.path.getsize(zip_filename) / (1024 * 1024):.1f} MB")
    
    return zip_filename

# Run this function after training both models
create_model_summary()

print("\nModel management functions defined successfully!")

Creating model summary...
No models directory found. Train models first.

Model management functions defined successfully!


## Local Loading Example (For Any GPU/CPU)

### Instructions for Local Use:

1. **Extract downloaded models**: Unzip `trained_models.zip` to get the `models/` folder
2. **Choose device wisely**: Use CPU or GPU based on memory availability
3. **Batch size adjustment**: The loading functions automatically adjust batch sizes for weak GPU

### Example Usage on Local Machine:

In [9]:
# Example: Load and test models locally (Run this after training)
import torch

# Choose device based on your local setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device for local inference: {device}")

if device.type == 'cuda':
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / (1024**3)  # GB
    print(f"GPU Memory Available: {gpu_memory:.1f} GB")

# Example of loading a trained model (uncomment after training)
"""
# Load ResNet model
resnet_path = "models/ResNet50_best_model.pth" 
resnet_metadata = "models/ResNet50_best_metadata.json"

if os.path.exists(resnet_path):
    loaded_resnet, metadata = load_model_for_inference(
        resnet_path, resnet_metadata, model_type="resnet", device=device
    )
    print("ResNet loaded successfully!")
else:
    print("ResNet model not found. Train the model first.")
"""

print("Local inference setup complete!")

Using device for local inference: cuda
GPU Memory Available: 2.0 GB
Local inference setup complete!
GPU Memory Available: 2.0 GB
Local inference setup complete!


### Memory Optimization Tips for Various Hardware:

1. **Batch Size**: The loading functions automatically use smaller batch sizes (8-16) for inference
2. **Model Choice**: ResNet-50 (23.5M params) is more memory-efficient than ViT (85.8M params)
3. **CPU Fallback**: If GPU memory is insufficient, the code automatically falls back to CPU
4. **Cache Clearing**: GPU cache is cleared after each batch to prevent memory buildup
5. **Gradient Disabled**: `torch.no_grad()` ensures no gradients are computed during inference

### Performance Expectations:
- **ResNet on limited GPU**: Should work well with reduced batch size
- **ViT on limited GPU**: May require CPU inference depending on available memory
- **CPU Inference**: Slower but will work for both models

## Model Evaluation and Comparison

Now we'll evaluate both trained models on the test set and compare their performance using comprehensive metrics:

1. **Test Set Evaluation**: Both models will be evaluated on the unseen test data
2. **Metrics Calculation**: Accuracy, Precision, Recall, F1-Score (both macro-averaged and per-class)
3. **Confusion Matrix**: Visual representation of prediction performance
4. **Comparative Analysis**: Side-by-side comparison of model performance

In [ ]:
# Evaluate both models on test set
print("Evaluating ResNet on test set...")
resnet_metrics = evaluate_model(resnet_model, test_loader, class_names)

print("\nEvaluating Vision Transformer on test set...")
vit_metrics = evaluate_model(vit_model, test_loader, class_names)

# Print detailed results for both models
print_detailed_results(resnet_metrics, "ResNet-50", class_names)
print_detailed_results(vit_metrics, "Vision Transformer", class_names)

In [ ]:
# Plot confusion matrices
plot_confusion_matrix(resnet_metrics['confusion_matrix'], class_names, "ResNet-50")
plot_confusion_matrix(vit_metrics['confusion_matrix'], class_names, "Vision Transformer")

# Compare metrics between models
plot_metrics_comparison(resnet_metrics, vit_metrics, class_names)

## 📊 Comparative Evaluation

Run the ResNet-50 and ViT training and evaluation cells to generate a comparison from the current code. The committed notebook does not contain completed test outputs, so no accuracy values are reported here.

Because LC25000 contains augmented derivatives and this workflow lacks source-image or patient identifiers, use a grouped split before interpreting results as generalization to unseen tissue sources.


In [ ]:
# Final Comparison Summary
print("\n" + "="*80)
print("FINAL COMPARISON SUMMARY")
print("="*80)

models_comparison = {
    'Model': ['ResNet-50', 'Vision Transformer'],
    'Test Accuracy': [resnet_metrics['accuracy'], vit_metrics['accuracy']],
    'Precision': [resnet_metrics['precision'], vit_metrics['precision']],
    'Recall': [resnet_metrics['recall'], vit_metrics['recall']],
    'F1-Score': [resnet_metrics['f1'], vit_metrics['f1']],
    'Parameters': ['23.5M', '85.8M']
}

import pandas as pd
comparison_df = pd.DataFrame(models_comparison)
print(comparison_df.to_string(index=False, float_format='%.4f'))

# Determine best model
best_model = "ResNet-50" if resnet_metrics['accuracy'] > vit_metrics['accuracy'] else "Vision Transformer"
print(f"\nBest performing model: {best_model}")

print("\nKey Observations:")
print("-" * 50)
if resnet_metrics['accuracy'] > vit_metrics['accuracy']:
    diff = resnet_metrics['accuracy'] - vit_metrics['accuracy']
    print(f"• ResNet-50 outperforms ViT by {diff:.4f} ({diff*100:.2f}%) in accuracy")
else:
    diff = vit_metrics['accuracy'] - resnet_metrics['accuracy']
    print(f"• Vision Transformer outperforms ResNet by {diff:.4f} ({diff*100:.2f}%) in accuracy")

print(f"• ResNet-50 has {(85.8-23.5):.1f}M fewer parameters than ViT")
print(f"• Both models show strong performance on this medical imaging task")
print(f"• The choice between models depends on accuracy vs. efficiency trade-offs")

## Project Summary

The notebook implements LC25000 loading, transfer-learning models, training loops, checkpoint helpers, and evaluation functions for ResNet-50 and ViT-Base. The recorded data counts are 18,000 training images, 4,501 validation images, and 2,499 test images.

The saved execution is incomplete and should be rerun after setting `data_root`. Report new metrics only after the full training and held-out test evaluation finish. For stronger evidence, reconstruct groups from the original source images or use another dataset with patient identifiers.